In [ ]:
using Pkg
Pkg.activate("..")

using OptimalControl
using Plots
using ForwardDiff
using DifferentialEquations
using MINPACK

### Optimal control problem

We consider the following optimal control problem 

$$
    \left\{ \begin{array}{ll}
    \displaystyle \min_{x,u} \int_{t_0}^{t_f} x(t) ~\mathrm dt \\[1em]
    \text{s.c.}~\dot x(t) = u(t), & t\in [t_0, t_f]~\mathrm{a.e.}, \\[0.5em]
    \phantom{\mathrm{s.c.}~} u(t) \in [-1,1], & t\in [t_0, t_f], \\[0.5em]
    \phantom{\mathrm{s.c.}~} x(t_0) = x_0, \quad x(t_f) = x_f,
    \end{array} \right.
$$

with $x_0$, $t_0$, $x_f$ and $t_f$ fixed. This problem is simple, and can be analytically solve without the use of numerical method. However, the goal is to solve this problem by indirect shooting.  

### Indirect method

We introduce the pseudo-Hamiltonian 

$$
    h(x,p,p^0,u) = p^0 x + p u.
$$

For the sake of simplicity, we consider in this notebook only the normal case, and we fix $p^0 = -1$. According to the Pontryagin maximum principle, the maximizing control is given by $u(x,p) \to \mathrm{sign}(p)$. This function is non-differentiable, and may lead to numerical issues.  



Let us import the necessary package and define the studied optimal control problem with some fixed initial and final time and state values.

In [ ]:
t0 = 0; x0 = 0; tf = 5; xf = 0          # Initial and final time and state

# Problem definition
@def ocp begin                          

    t ∈ [ t0, tf ], time
    x ∈ R, state
    u ∈ R, control

    x(t0) == x0
    x(tf) == xf

    ẋ(t) == u(t)      

    ∫( x(t) ) → min

end

Thanks to the control-toolbox, the flow $\varphi$ of the (true) Hamiltonian

$$
    H(x,p) = h(x,p,-1, u(x,p)) = p^0 x + \lvert p \rvert 
$$

 is given by the function $\texttt{Flow}$. The shooting function $S \colon \mathbb{R} \to \mathbb{R}$ is defined by

 $$
    S(p_0) = \pi \big( \varphi(t_0, x_0, p_0, t_f) \big) - x_f
 $$ 

 where $\pi (x,p) = x$ is the classical $x$-space projection.


In [ ]:
ϕ = Flow(ocp, (x,p) -> sign(p))             # Flow with maximizing control 
π((x,p)) = x;                               # Projection on state space

S(p0) = π( ϕ(t0, x0, p0, tf) ) - xf;        # Shooting function
nle = p0 -> [S(p0[1])]                      # Intermediate function

# Plot
plt = plot(xlim = (-7, 2), ylim = (-6, 6), size=(800, 500))
plot!(plt, S, label = "S")
plot!(plt, [-7,2], [0,0], c = :black, ls = :dash, label = nothing)

The main goal now is to find the zero of $S$. To this purpose, we use the numerical solver $\texttt{hybrd1}$ given in the package $\texttt{MINPACK.jl}$. If we don't provide the Jacobian $J_S$ of $S$ to the solver, the finite difference method is used to approximate it. 


In [ ]:
ξ = [-1.0]                                          # Initial guess
S!(s, ξ) = (s[:] .= S(ξ[1]); nothing)               # Intermediate function
p0_sol = fsolve(S!, ξ, show_trace = true)           # Solve

In [ ]:
sol = ϕ((t0, tf), x0, p0_sol.x)                     # Get the optimal trajectory
plot(sol, size = (800,600))                         # Plot

Now, we want to provide $J_S$ to the solver, thanks to the $\texttt{ForwardDiff.jl}$ package. This Jacobian is computed with the variational equation, and leads to a false result in our case.

Details.

Denoting $z = (x,p)$, we have 

$$
    \varphi(t_0, z_0, t_f) = z_0 + \int_{t_0}^{t_f} \vec H\big(\varphi(t_0, z_0, t)\big) \,\mathrm dt. 
$$

If we assume that $z_0 \to \varphi(t_0, z_0, t_f)$ is differentiable, we have  

$$
    \frac{\partial \varphi}{\partial z_0}(t_0, z_0, t_f)\cdot \delta z_0 =  \delta z_0 + \int_{t_0}^{t_f} \vec H'\big(\varphi(t_0, z_0, t)\big)\cdot \left( \frac{\partial \varphi}{\partial z_0}(t_0, z_0, t) \cdot \delta z_0 \right) \,\mathrm dt, 
$$

and so, $z_0 \to \frac{\partial \varphi}{\partial z_0}(t_0, z_0, t_f)\cdot \delta z_0$ is solution of the variational equations

$$
    \frac{\partial \delta z}{\partial t}(t) = \vec H'\big(\varphi(t_0, z_0, t_f)\big) \cdot \delta z(t), \qquad \delta z(t_0) = \delta z_0.
$$

In the studied optimal control problem, we have 

$$
    \vec H(x,p) = (\mathrm{sign}(p), -1) 
$$

and so, we have $\vec H'(z) = 0_2$ almost everywhere, which implies


$$
    \frac{\partial \varphi}{\partial z_0}(t_0, z_0, t_f) \cdot \delta z_0 = \mathrm{exp}\big((t_f-t_0) 0_2 \big)\cdot \delta z_0 = \delta z_0.
$$ 

The Jacobian of the shooting function is then given by 

$$
    S'(p_0) = \pi \left( \frac{\partial \varphi}{\partial p_0}(t_0, x_0, p_0, t_f) \right) = \pi \left( \frac{\partial \varphi}{\partial z_0}(t_0, z_0, t_f) \cdot (0,1) \right) = \pi(0,1) = 0. 
$$

In [ ]:
ξ = [-1.0]                                          # Initial guess
JS(ξ) = ForwardDiff.jacobian(p -> [S(p[1])], ξ)     # Compute jacobian by forward differentiation
println("ξ = ", ξ[1])
println("JS(ξ) : ", JS(ξ)[1])

However, the solver $\texttt{hybrd1}$ uses rank 1 approximation to actualize the Jacobian insted of compute it at each iteration, which imply that it still converges to the solution even if the given Jacobian is completely false.

In [ ]:
JS!(js, ξ) = (js[:] .= JS(ξ); nothing)          # Intermediate function
p0_sol = fsolve(S!, JS!, ξ, show_trace = true)  # Solve

In [ ]:
sol = ϕ((t0, tf), x0, p0_sol.x)                 # Get the optimal trajectory
plt = plot(sol, size = (800, 600))              # Plot

The goal is to provide the true Jacobian of $S$ by using the $\texttt{ForwardDiff}$ package, and so we need to indicate to the solver that the dynamic of the system change when $p = 0$.

To understang why we need to give this information to the solver, see the following details. 

Details. 

The problem is that the Hamiltonian $H$ is not differentiable everywhere due to the maximizing control. This control is bang-bang ($u = 1$ and $u = -1$). 
 
Let now construct the two smooth Hamiltonians associated to these two controls 

$$
    H^+(x,p) = h(x,p,-1,1) = -x + p \qquad \text{and} \qquad H^-(x,p) = h(x,p,-1,-1) =  -x - p. 
$$

Their associated vector fields are given by 

$$
    \vec H^+(x,p) = (1,1) \qquad \text{and} \qquad \vec H^-(x,p) = (-1, 1), 
$$

and their associated flow correspond to 

$$
    \varphi^+(t_0, z_0, t_f) = z_0 + \left( \begin{array}{c} 1 \\ 1 \end{array} \right) (t_f -t_0)
    \qquad \text{and} \qquad 
    \varphi^-(t_0, z_0, t_f) = z_0 + \left( \begin{array}{c} -1 \\ \phantom{-} 1 \end{array} \right) (t_f -t_0).
$$

If we assume that the optimal structure of the problem is negative then positive bangs, then the associated flow is defined by  

$$ 
    \varphi(t_0, z_0, t_f) = \varphi^+ \big( t_1(z_0), \varphi^-\big(t_0, z_0, t_1(z_0)\big), t_f \big),
$$

with the following condition 

$$
    \pi_p \big( \varphi^-(t_0, z_0, t_1(z_0)) \big) = 0,
$$

where $\pi_p(x,p) = p$ is the classical $p$-space projection. By devlopping this last condition, an explicit form of the function $t_1(\cdot)$ is given by 

$$
    t_1(x_0, p_0) = t_0 - p_0.
$$

Finally, we have 

$$
    \begin{align*}
    \frac{\partial \varphi}{\partial z_0} 
    &= \frac{\partial \varphi^+}{\partial t_0} \frac{\partial t_1}{\partial z_0} + \frac{\varphi^+}{\partial z_0} \left( \frac{\partial \varphi^-}{\partial z_0} + \frac{\partial \varphi^-}{\partial t_f} \frac{\partial t_1}{\partial z_0} \right) \\
    &= \left( \begin{array}{c} -1 \\ -1 \end{array} \right) \left( \begin{array}{cc}0 & -1 \end{array} \right) + \left( \begin{array}{cc} 1 & 0 \\ 0 & 1 \end{array} \right) \left[
    \left( \begin{array}{cc} 1 & 0 \\ 0 & 1 \end{array} \right) + \left( \begin{array}{c} -1 \\ \phantom - 1 \end{array} \right) \left( \begin{array}{cc}0 & -1 \end{array} \right)
    \right] \\
    &= \left( \begin{array}{cc} 0 & 1 \\ 0 & 1 \end{array} \right) + \left( \begin{array}{cc} 1 & 0 \\ 0 & 1 \end{array} \right) + \left( \begin{array}{cc} 0 & \phantom -1 \\ 0 & -1 \end{array} \right) \\ 
    &= \left( \begin{array}{cc} 1 & 2 \\ 0 & 1 \end{array} \right)
    \end{align*}
$$

and so, we have that 

$$
    S'(p_0) = \pi \left( \frac{\partial \varphi}{\partial p_0}(t_0, x_0, p_0, t_f) \right) = \pi \left( \frac{\partial \varphi}{\partial z_0}(t_0, z_0, t_f) \cdot (0,1) \right) = \pi(2,1) = 2.
$$

End of details

To provide this change of dynamic to the solver, we need to use a callback during the integration that will execute the function $\texttt{affect!}$ when $\texttt{condition(x,p)} = 0$.

For us, the condition is given by $(x,p) \to p$. For the $\texttt{affect!}$ function, we use a global parameter $\alpha$. This parameter will be set to $\pm 1$ at the beginning of the integration and it sign will change with the $\texttt{affect!}$ function. 

Thanks to the $\texttt{control-toolbox}$ package, the created callback can be easily pass to the integrator throught the $\texttt{Flow}$ function.

In [ ]:
# Parameter: ̇p(t) = α with α = ±1
global α

# Event when condition(x,p) == 0
function condition(z, t, integrator)                        
    x,p = z
    return p
end

# Action when condition == 0 
function affect!(integrator)
    global α = -α
    nothing
end

cb = ContinuousCallback(condition, affect!)                 # Callback 
φ_ = Flow(ocp, (x,p) -> α, callback = cb)                   # Intermediate flow

# Flow for the solver
function φ(t0, x0, p0, tf; kwargs...)                       
    global α = sign(p0)
    return φ_(t0, x0, p0, tf; kwargs...)
end

# Flow for plot
function φ((t0, tf), x0, p0; kwargs...)                     
    global α = sign(p0)
    return φ_((t0, tf), x0, p0; kwargs...)
end

# Plot solution
function plot_sol(sol; size = (800, 600))
    
    t = sol.time_grid.value
    x = state(sol)
    p = costate(sol)
    u = sign ∘ p

    plt_x = plot(t, x, label = nothing, ylabel = "x", title = "state", titlefontsize = 10, lw = 2)
    plt_p = plot(t, p, label = nothing, title = "costate", titlefontsize = 10, lw = 2)
    plt_u = plot(t, u, label = nothing, ylabel = "u", title = "control", titlefontsize = 10, lw = 2)

    plt_xp = plot(plt_x, plt_p, layout=(1, 2))
    return plot(plt_xp, plt_u, layout = (2, 1), size = size)
end

nothing; # hide

In [ ]:
Shoot(p0) = π( φ(t0, x0, p0, tf) ) - xf                     # Shooting function
ξ = [-1.0]                                                  # Initial guess
JShoot(ξ) = ForwardDiff.jacobian(p -> [Shoot(p[1])], ξ)     # Compute jacobian by forward differentiation
println("ξ = ", ξ[1])
println("JS(ξ) : ", JShoot(ξ)[1])

In [ ]:
Shoot!(shoot, ξ) = (shoot[:] .= Shoot(ξ[1]); nothing)       # Intermediate function
JShoot!(jshoot, ξ) = (jshoot[:] .= JShoot(ξ); nothing)      # Intermediate function
p0_sol = fsolve(Shoot!, JShoot!, ξ, show_trace = true)      # Solve

In [ ]:
sol = φ((t0, tf), x0, p0_sol.x[1], saveat=range(t0, tf, 500))  # Get optimal trajectory
plot_sol(sol)